# SHROOM RAG E5 — Module 2 Phase B Evaluation

Στόχος είναι η αυστηρά offline αξιολόγηση των έτοιμων Phase A candidates με τις κλειδωμένες ανθρώπινες relevance annotations. Δεν εκτελείται retrieval, dense encoding, official test, API ή download.

Η `Q_SRC_TGT` είναι **reference-assisted** και δεν αποτελεί ανεξάρτητη external validation. Η επιλογή gate είναι προκαταρκτική development επιλογή πάνω σε μόνο 10 examples.


In [ ]:
# 1. Setup και σταθερές αναπαραγωγιμότητας
from __future__ import annotations
import csv, hashlib, json, math, os, pathlib, statistics, tempfile
from collections import Counter, defaultdict

RANDOM_STATE = 42

# Προσαρμοσμένος κώδικας και ιστορικά τεκμήρια έχουν χωριστές ταυτότητες.
import importlib.util as _module2_import
import sys as _module2_sys
from pathlib import Path as _Module2Path
_module2_sys.dont_write_bytecode = True
_module2_code_root = _Module2Path(os.environ["SHROOM_MODULE2_CODE_ROOT"])
if not _module2_code_root.is_absolute() or ".." in _module2_code_root.parts:
    raise ValueError("SHROOM_MODULE2_CODE_ROOT must be an absolute local directory.")
_module2_helper_path = _module2_code_root / "scripts" / "module2_dense_cache.py"
for _part in (_module2_helper_path, *_module2_helper_path.parents):
    if _part.is_symlink() or (getattr(_part.lstat(), "st_file_attributes", 0) & 0x400):
        raise ValueError("Reparse/symlink helper paths are not accepted.")
_module2_raw = _module2_helper_path.read_bytes()
if hashlib.sha256(_module2_raw).hexdigest() != os.environ["SHROOM_MODULE2_HELPER_SHA256"].lower():
    raise ValueError("Adapted helper pin mismatch; no fallback to historical source.")
if "module2_dense_cache" in _module2_sys.modules:
    _module2 = _module2_sys.modules["module2_dense_cache"]
    if _Module2Path(_module2.__file__) != _module2_helper_path or getattr(_module2, "_notebook_pin", None) != hashlib.sha256(_module2_raw).hexdigest():
        raise ValueError("A different/stale helper is already loaded; restart the kernel.")
else:
    _spec = _module2_import.spec_from_file_location("module2_dense_cache", _module2_helper_path)
    _module2 = _module2_import.module_from_spec(_spec)
    _module2_sys.modules["module2_dense_cache"] = _module2
    _spec.loader.exec_module(_module2)
    _module2._notebook_pin = hashlib.sha256(_module2_raw).hexdigest()
CTX = _module2.context_from_environment(require_cache=False)
print("Adapted Module 2: H0 evidence guards retained; runtime reproduction NOT VERIFIED.")

ROOT = CTX.evidence("RAG")
PHASE_A = ROOT / "results" / "rag_module2"
ANNOTATION_DIR = PHASE_A / "annotation"
EVAL_DIR = PHASE_A / "evaluation"
PATHS = {
    "human": ANNOTATION_DIR / "module2_annotation_dev_human_completed.csv",
    "blinded": ANNOTATION_DIR / "module2_annotation_dev_blinded.csv",
    "provenance": ANNOTATION_DIR / "module2_annotation_provenance.jsonl",
    "pool_manifest": ANNOTATION_DIR / "module2_annotation_pool_manifest.json",
    "candidates": PHASE_A / "module2_candidates_top5.jsonl",
    "phase_a_manifest": PHASE_A / "module2_input_manifest.json",
}
OUTPUTS = {
    "metrics": CTX.run_dir / "evaluation" / "module2_dev_retrieval_metrics.csv",
    "sensitivity": CTX.run_dir / "evaluation" / "module2_dev_gate_sensitivity.csv",
    "decisions": CTX.run_dir / "evaluation" / "module2_dev_decisions.jsonl",
    "selected": CTX.run_dir / "evaluation" / "module2_selected_retrieval_config.json",
    "report": CTX.run_dir / "evaluation" / "MODULE2_PHASE_B_RESULTS.md",
}
EXPECTED_HUMAN_SHA256 = "3f1f0a6d3eb9a75de69344f6e1df13b88745e464c98f94e1ce87997302669ed5"
METHODS = ("exact", "bm25", "dense", "rrf")
QUERY_VARIANTS = ("Q_SRC", "Q_SRC_TGT")
STRICT = {"relevant_same_sense"}
LENIENT = {"relevant_same_sense", "partially_relevant"}
UNSAFE = {"different_sense", "irrelevant"}
ALLOWED_CATEGORIES = STRICT | LENIENT | UNSAFE | {"uncertain_review"}
ALLOWED_CONFIDENCES = {"high", "medium", "low"}
REQUIRED_COLUMNS = ["annotation_id","example_id","term","src","tgt","evidence_id","candidate_term","candidate_definition","candidate_context","relevance_category","annotation_confidence","annotation_notes"]
FORBIDDEN_FIELDS = {"label", "labels", "label_bin", "p(Hallucination)", "p_hallucination"}

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("Python offline evaluation; RANDOM_STATE=42; network/API/model execution disabled.")


## 2. Preflight

Το preflight σταματά πριν από οποιαδήποτε εγγραφή αν αποτύχει hash, annotation integrity, blinded-column equality ή stable-ID join. Το provenance χρησιμοποιείται μόνο για να επανασυνδέσει τα blinded IDs με τα ήδη υπάρχοντα Phase A diagnostics.


In [ ]:
# 2. Βοηθητικές συναρτήσεις και read-only φόρτωση
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest().lower()

def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        return list(reader), list(reader.fieldnames or [])

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

human_rows, human_fields = read_csv(PATHS["human"])
blinded_rows, blinded_fields = read_csv(PATHS["blinded"])
provenance_rows = read_jsonl(PATHS["provenance"])
candidate_rows = read_jsonl(PATHS["candidates"])
pool_manifest = json.loads(PATHS["pool_manifest"].read_text(encoding="utf-8"))
phase_a_manifest = json.loads(PATHS["phase_a_manifest"].read_text(encoding="utf-8"))

protected_paths = {
    "phase_a_notebook": ROOT / "shroom_rag_e5_module2_retrieval.ipynb",
    "dense_cache_helper": ROOT / "scripts" / "module2_dense_cache.py",
    "module1_trial_notebook": ROOT / "shroom_rag_e5_module1_gemini_trial.ipynb",
    "module1_notebook": ROOT / "shroom_rag_e5_module1.ipynb",
    "e1_e4_notebook": ROOT / "shroom_ensemble.ipynb",
}
protected_before = {name: sha256(path) for name, path in protected_paths.items()}


In [ ]:
# 3. Υποχρεωτικό preflight και stable-ID join
checks = []
def require(name, condition, detail=""):
    checks.append({"check": name, "passed": bool(condition), "detail": str(detail)})
    if not condition:
        raise AssertionError(f"PREFLIGHT FAILED: {name}: {detail}")

require("83 rows x 12 columns", len(human_rows) == 83 and human_fields == REQUIRED_COLUMNS, (len(human_rows), len(human_fields)))
annotation_ids = [r["annotation_id"] for r in human_rows]
require("83 unique nonempty annotation_id", len(annotation_ids) == len(set(annotation_ids)) == 83 and all(annotation_ids))
require("mandatory annotations complete", all(r["relevance_category"] and r["annotation_confidence"] for r in human_rows))
require("exactly 10 development examples", len({r["example_id"] for r in human_rows}) == 10)
require("allowed relevance categories", {r["relevance_category"] for r in human_rows} <= ALLOWED_CATEGORIES, Counter(r["relevance_category"] for r in human_rows))
require("allowed annotation confidences", {r["annotation_confidence"] for r in human_rows} <= ALLOWED_CONFIDENCES, Counter(r["annotation_confidence"] for r in human_rows))
require("first 9 columns exactly match blinded CSV", human_fields == blinded_fields and len(human_rows) == len(blinded_rows) and all([a[k] for k in REQUIRED_COLUMNS[:9]] == [b[k] for k in REQUIRED_COLUMNS[:9]] for a,b in zip(human_rows, blinded_rows)))
require("human annotation SHA-256", sha256(PATHS["human"]) == EXPECTED_HUMAN_SHA256, sha256(PATHS["human"]))

provenance_by_id = {r["annotation_id"]: r for r in provenance_rows}
require("provenance IDs unique", len(provenance_by_id) == len(provenance_rows) == 179)
require("human-to-provenance full join", all(r["annotation_id"] in provenance_by_id for r in human_rows))
require("development split only", {provenance_by_id[r["annotation_id"]]["split"] for r in human_rows} == {"retrieval-development"})
require("evidence IDs preserved", all(str(provenance_by_id[r["annotation_id"]]["evidence_id"]) == r["evidence_id"] for r in human_rows))
development_indices = sorted({provenance_by_id[r["annotation_id"]]["development_index"] for r in human_rows})
require("locked 10-example split", development_indices == pool_manifest["split_manifest"]["retrieval_development_indices"], development_indices)

candidate_by_identity = {(r["development_index"], r["query_variant"], r["retrieval_method"], r["method_rank"], r["evidence_id"]): r for r in candidate_rows}
require("Phase A identity keys unique", len(candidate_by_identity) == len(candidate_rows) == 612)
joined_appearances = 0
for annotation in human_rows:
    provenance = provenance_by_id[annotation["annotation_id"]]
    for appearance in provenance["appearances"]:
        key = (provenance["development_index"], appearance["query_variant"], appearance["retrieval_method"], appearance["method_rank"], provenance["evidence_id"])
        require("appearance joins exactly once", key in candidate_by_identity, key)
        joined_appearances += 1
require("all development appearances joined", joined_appearances == 192, joined_appearances)

all_input_fields = set(human_fields) | set(candidate_rows[0]) | set(provenance_rows[0])
require("forbidden labels/probabilities absent", not (FORBIDDEN_FIELDS & all_input_fields), sorted(FORBIDDEN_FIELDS & all_input_fields))
require("official-test paths absent", all("official test" not in str(path).lower() and "labeled test" not in str(path).lower() for path in PATHS.values()))

bad_phase_a = []
for relative, expected in pool_manifest["phase_a_tree_before_creation"]["files"].items():
    path = CTX.evidence("RAG/results/rag_module2/" + relative)
    if not path.exists() or sha256(path) != expected.lower():
        bad_phase_a.append(relative)
require("43 locked Phase A artifacts unchanged", not bad_phase_a, bad_phase_a)
require("Phase A notebook unchanged", protected_before["phase_a_notebook"] == pool_manifest["input_file_hashes"]["phase_a_notebook_sha256"].lower())
require("dense helper unchanged", protected_before["dense_cache_helper"] == pool_manifest["input_file_hashes"]["dense_cache_helper_sha256"].lower())
require("Module 1 trial unchanged", protected_before["module1_trial_notebook"] == pool_manifest["input_file_hashes"]["module1_trial_notebook_sha256"].lower())
print(f"PREFLIGHT PASS — {len(human_rows)} annotations, 10 examples, {joined_appearances} joined appearances, {len(checks)} checks")


## 3. Metric definitions

- **Strict relevant:** `relevant_same_sense`.
- **Lenient relevant:** strict + `partially_relevant`.
- **Unsafe:** `different_sense` + `irrelevant`.
- `uncertain_review` μένει ξεχωριστό.
- Relevant@1 έχει παρονομαστή και τα 10 development examples· απουσία result μετρά ως μη relevant.
- Precision@k είναι macro precision στις έως k ανακτημένες εγγραφές. Επιστρέφει `NA` αν υπάρχει ανακτημένη αλλά μη annotated θέση.
- Πλήρες MRR επιστρέφεται μόνο αν όλες οι ανακτημένες θέσεις είναι annotated. Το MRR@3 αναφέρεται χωριστά ως truncated diagnostic.


In [ ]:
# 4. Annotation lookup και deterministic metric functions
annotation_by_candidate = {}
annotation_meta = {}
term_by_index = {}
for row in human_rows:
    p = provenance_by_id[row["annotation_id"]]
    key = (p["development_index"], int(row["evidence_id"]))
    if key in annotation_by_candidate:
        raise AssertionError(f"Duplicate joined candidate annotation: {key}")
    annotation_by_candidate[key] = row["relevance_category"]
    annotation_meta[key] = {"annotation_id": row["annotation_id"], "confidence": row["annotation_confidence"], "notes": row["annotation_notes"]}
    term_by_index[p["development_index"]] = row["term"]

dev_candidates = [r for r in candidate_rows if r["development_index"] in development_indices]
ranked = defaultdict(list)
for row in dev_candidates:
    ranked[(row["query_variant"], row["retrieval_method"], row["development_index"])].append(row)
for key in ranked:
    ranked[key].sort(key=lambda r: (r["method_rank"], r["evidence_id"]))
    assert [r["method_rank"] for r in ranked[key]] == sorted(r["method_rank"] for r in ranked[key])

def relevant_at_1(query_variant, method, relevant_set):
    hits = 0
    for idx in development_indices:
        rows = ranked.get((query_variant, method, idx), [])
        if rows and annotation_by_candidate.get((idx, rows[0]["evidence_id"])) in relevant_set:
            hits += 1
    return hits / len(development_indices)

def precision_at_k(query_variant, method, k, relevant_set):
    values = []
    for idx in development_indices:
        rows = ranked.get((query_variant, method, idx), [])[:k]
        if any((idx, r["evidence_id"]) not in annotation_by_candidate for r in rows):
            return None, False
        values.append(sum(annotation_by_candidate[(idx, r["evidence_id"])] in relevant_set for r in rows) / len(rows) if rows else 0.0)
    return statistics.fmean(values), True

def reciprocal_rank(query_variant, method, relevant_set, limit=None):
    values, complete = [], True
    for idx in development_indices:
        rows = ranked.get((query_variant, method, idx), [])
        if limit is not None:
            rows = rows[:limit]
        if any((idx, r["evidence_id"]) not in annotation_by_candidate for r in rows):
            complete = False
            continue
        rr = next((1.0 / r["method_rank"] for r in rows if annotation_by_candidate[(idx, r["evidence_id"])] in relevant_set), 0.0)
        values.append(rr)
    return (statistics.fmean(values) if complete and len(values) == len(development_indices) else None), complete

def fmt(value):
    return "NA" if value is None else f"{value:.6f}"


In [ ]:
# 5. Retrieval metrics ανά method/query variant
metric_rows = []
for query_variant in QUERY_VARIANTS:
    for method in METHODS:
        top_labels = []
        for idx in development_indices:
            rows = ranked.get((query_variant, method, idx), [])
            top_labels.append(annotation_by_candidate[(idx, rows[0]["evidence_id"])] if rows else None)
        row = {"query_variant": query_variant, "reference_assisted": query_variant == "Q_SRC_TGT", "retrieval_method": method, "n_examples": 10, "top1_result_coverage": sum(x is not None for x in top_labels) / 10}
        row["strict_relevant_at_1"] = relevant_at_1(query_variant, method, STRICT)
        row["lenient_relevant_at_1"] = relevant_at_1(query_variant, method, LENIENT)
        for k in range(1, 6):
            strict_p, strict_full = precision_at_k(query_variant, method, k, STRICT)
            lenient_p, lenient_full = precision_at_k(query_variant, method, k, LENIENT)
            row[f"strict_precision_at_{k}"] = strict_p
            row[f"lenient_precision_at_{k}"] = lenient_p
            row[f"annotation_complete_at_{k}"] = strict_full and lenient_full
        row["strict_mrr"], row["strict_mrr_fully_defined"] = reciprocal_rank(query_variant, method, STRICT)
        row["lenient_mrr"], row["lenient_mrr_fully_defined"] = reciprocal_rank(query_variant, method, LENIENT)
        row["strict_mrr_at_3"], _ = reciprocal_rank(query_variant, method, STRICT, 3)
        row["lenient_mrr_at_3"], _ = reciprocal_rank(query_variant, method, LENIENT, 3)
        row["unsafe_top1_rate"] = sum(x in UNSAFE for x in top_labels) / 10
        row["different_sense_top1_rate"] = sum(x == "different_sense" for x in top_labels) / 10
        row["irrelevant_top1_rate"] = sum(x == "irrelevant" for x in top_labels) / 10
        row["uncertain_top1_rate"] = sum(x == "uncertain_review" for x in top_labels) / 10
        row["evidence_eligible_coverage_at_3"] = sum(any(annotation_by_candidate.get((idx,r["evidence_id"])) in LENIENT for r in ranked.get((query_variant,method,idx),[])[:3]) for idx in development_indices) / 10
        row["raw_abstention_rate"] = sum(x is None for x in top_labels) / 10
        row["raw_irrelevant_evidence_rate"] = row["unsafe_top1_rate"]
        metric_rows.append(row)

for row in metric_rows:
    print(row["query_variant"], row["retrieval_method"], "strict@1", fmt(row["strict_relevant_at_1"]), "lenient@1", fmt(row["lenient_relevant_at_1"]), "unsafe@1", fmt(row["unsafe_top1_rate"]))


## 4. Deterministic gate sensitivity

Το grid δεν δημιουργεί composite relevance score. Κάθε candidate περνά μόνο αν ικανοποιεί ταυτόχρονα ανεξάρτητους, ερμηνεύσιμους guards. Τα score και margin thresholds είναι `none`, 25ο, 50ό ή 75ο percentile των top-1 development retrieval signals. Το label χρησιμοποιείται μόνο αφού παρθεί η pass/abstain απόφαση.

Tie-breaking: μικρότερο unsafe rate στους accepted, μεγαλύτερη lenient usable coverage, μικρότερο unsafe rate σε όλα τα examples, μεγαλύτερη strict usable coverage, χαμηλότερη συνολική evidence coverage (συντηρητικότητα), τέλος lexicographic `configuration_id`.


In [ ]:
# 6. Gate signals, grid και Pareto frontier
SCORE_FIELD = {"exact": None, "bm25": "bm25_score", "dense": "dense_cosine_score", "rrf": "rrf_score"}

def quantile(values, q):
    values = sorted(float(v) for v in values)
    if not values:
        return None
    pos = (len(values)-1)*q
    lo, hi = math.floor(pos), math.ceil(pos)
    return values[lo] if lo == hi else values[lo]*(hi-pos) + values[hi]*(pos-lo)

def signals_for(query_variant, method):
    output = {}
    field = SCORE_FIELD[method]
    for idx in development_indices:
        rows = ranked.get((query_variant, method, idx), [])
        if not rows:
            output[idx] = None
            continue
        top = rows[0]
        score = float(top[field]) if field and top[field] is not None else None
        second = float(rows[1][field]) if field and len(rows)>1 and rows[1][field] is not None else None
        relative_margin = ((score-second)/max(abs(score),1e-12)) if score is not None and second is not None else 0.0
        duplicate = len({r["evidence_id"] for r in rows}) != len(rows)
        output[idx] = {"candidate": top, "score": score, "relative_margin": relative_margin, "method_agreement": len(top["retrieval_methods"]), "exact_match": bool(top["exact_match"]), "duplicate": duplicate}
    return output

def evaluate_gate(query_variant, method, score_label, score_threshold, margin_label, margin_threshold, min_agreement, require_exact, reject_duplicates=True):
    sig = signals_for(query_variant, method)
    decisions = []
    for idx in development_indices:
        s = sig[idx]
        reasons = []
        if s is None: reasons.append("no_candidate")
        else:
            if score_threshold is not None and (s["score"] is None or s["score"] < score_threshold): reasons.append("score_below_threshold")
            if margin_threshold is not None and s["relative_margin"] < margin_threshold: reasons.append("margin_below_threshold")
            if s["method_agreement"] < min_agreement: reasons.append("insufficient_method_agreement")
            if require_exact and not s["exact_match"]: reasons.append("exact_term_required")
            if reject_duplicates and s["duplicate"]: reasons.append("duplicate_candidate")
        accepted = not reasons
        label = annotation_by_candidate[(idx, s["candidate"]["evidence_id"])] if s else None
        decisions.append({"development_index": idx, "accepted": accepted, "reasons": reasons, "label": label, "signals": s})
    accepted = [d for d in decisions if d["accepted"]]
    n = len(development_indices)
    unsafe = sum(d["label"] in UNSAFE for d in accepted)
    strict = sum(d["label"] in STRICT for d in accepted)
    lenient = sum(d["label"] in LENIENT for d in accepted)
    uncertain = sum(d["label"] == "uncertain_review" for d in accepted)
    config_id = f"{query_variant}|{method}|score={score_label}|margin={margin_label}|agree={min_agreement}|exact={int(require_exact)}|dedup=1"
    return {"configuration_id":config_id,"query_variant":query_variant,"reference_assisted":query_variant=="Q_SRC_TGT","retrieval_method":method,"score_threshold_rule":score_label,"score_threshold":score_threshold,"relative_margin_threshold_rule":margin_label,"relative_margin_threshold":margin_threshold,"min_method_agreement":min_agreement,"require_exact_match":require_exact,"reject_duplicates":True,"accepted_count":len(accepted),"evidence_coverage":len(accepted)/n,"strict_usable_coverage":strict/n,"lenient_usable_coverage":lenient/n,"unsafe_count":unsafe,"unsafe_evidence_rate_accepted":unsafe/len(accepted) if accepted else None,"unsafe_evidence_rate_all":unsafe/n,"uncertain_accepted_count":uncertain,"abstention_rate":1-len(accepted)/n,"all_abstain":not accepted,"pareto_frontier":False,"selected":False,"decisions":decisions}

sensitivity_rows = []
for qv in QUERY_VARIANTS:
    for method in METHODS:
        signal_map = signals_for(qv, method)
        score_values = [s["score"] for s in signal_map.values() if s and s["score"] is not None]
        margin_values = [s["relative_margin"] for s in signal_map.values() if s]
        score_grid = [("none",None)] if not score_values else [("none",None)]+[(f"q{int(q*100):02d}",quantile(score_values,q)) for q in (.25,.50,.75)]
        margin_grid = [("none",None),("zero",0.0)]+[(f"q{int(q*100):02d}",quantile(margin_values,q)) for q in (.25,.50,.75)]
        for sl,st in score_grid:
            for ml,mt in margin_grid:
                for agreement in (1,2,3):
                    for exact_required in (False,True):
                        sensitivity_rows.append(evaluate_gate(qv,method,sl,st,ml,mt,agreement,exact_required))

def dominates(a,b):
    weak = a["lenient_usable_coverage"] >= b["lenient_usable_coverage"] and a["unsafe_evidence_rate_all"] <= b["unsafe_evidence_rate_all"] and a["abstention_rate"] <= b["abstention_rate"]
    strict = a["lenient_usable_coverage"] > b["lenient_usable_coverage"] or a["unsafe_evidence_rate_all"] < b["unsafe_evidence_rate_all"] or a["abstention_rate"] < b["abstention_rate"]
    return weak and strict

nontrivial = [r for r in sensitivity_rows if not r["all_abstain"]]
for row in nontrivial:
    row["pareto_frontier"] = not any(dominates(other,row) for other in nontrivial if other is not row)
pareto_rows = [r for r in nontrivial if r["pareto_frontier"]]

eligible = [r for r in sensitivity_rows if r["accepted_count"] >= 2 and r["lenient_usable_coverage"] >= .20 and (r["unsafe_evidence_rate_accepted"] is not None and r["unsafe_evidence_rate_accepted"] <= .10)]
eligible.sort(key=lambda r:(r["unsafe_evidence_rate_accepted"],-r["lenient_usable_coverage"],r["unsafe_evidence_rate_all"],-r["strict_usable_coverage"],r["evidence_coverage"],r["configuration_id"]))
selected_gate = eligible[0] if eligible else None
selection_status = "PRELIMINARY_DEVELOPMENT_SELECTION" if selected_gate else "NO_GO_FOR_HOLDOUT"
if selected_gate: selected_gate["selected"] = True
print("Grid configurations:",len(sensitivity_rows),"Pareto:",len(pareto_rows),"Status:",selection_status)
print("Selected:",selected_gate["configuration_id"] if selected_gate else "NO_GO_FOR_HOLDOUT")


## 5. Deterministic tests

Τα synthetic tests δεν χρησιμοποιούν project annotations για να ορίσουν το αναμενόμενο αποτέλεσμα. Οι τελευταίοι guards επαναϋπολογίζουν τα hashes των κλειδωμένων artifacts μετά την εγγραφή των νέων Phase B outputs.


In [ ]:
# 7. Offline deterministic tests
test_results = []
def test(name, fn):
    try:
        fn(); test_results.append((name, True, ""))
    except Exception as exc:
        test_results.append((name, False, str(exc)))

def assert_true(x): assert x
test("annotation integrity and hash", lambda: assert_true(len(human_rows)==83 and sha256(PATHS["human"])==EXPECTED_HUMAN_SHA256))
test("complete ID join", lambda: assert_true(joined_appearances==192 and len(annotation_by_candidate)==83))
test("duplicate/missing IDs", lambda: assert_true(len(annotation_ids)==len(set(annotation_ids)) and all(annotation_ids)))
test("metric synthetic strict@1", lambda: assert_true((sum(x in STRICT for x in ["relevant_same_sense","irrelevant"])/2)==0.5))
test("metric synthetic unsafe", lambda: assert_true(sum(x in UNSAFE for x in ["different_sense","partially_relevant","irrelevant"])==2))
test("stable ranking tie-break", lambda: assert_true(sorted([{"method_rank":1,"evidence_id":2},{"method_rank":1,"evidence_id":1}],key=lambda r:(r["method_rank"],r["evidence_id"]))[0]["evidence_id"]==1))
test("gate determinism", lambda: assert_true([{k:v for k,v in r.items() if k!="decisions"} for r in sensitivity_rows] == [{k:v for k,v in r.items() if k!="decisions"} for r in sensitivity_rows]))
test("abstention behavior", lambda: assert_true(all(r["accepted_count"]==0 and r["abstention_rate"]==1 for r in sensitivity_rows if r["all_abstain"])))
test("forbidden labels/probabilities absent", lambda: assert_true(not(FORBIDDEN_FIELDS & all_input_fields)))
test("official test absent", lambda: assert_true(all("official test" not in str(p).lower() and "labeled test" not in str(p).lower() for p in PATHS.values())))
test("Phase A protected tree unchanged", lambda: assert_true(all(sha256(CTX.evidence("RAG/results/rag_module2/" + rel))==expected.lower() for rel,expected in pool_manifest["phase_a_tree_before_creation"]["files"].items())))
test("Module 1/E1-E4 unchanged in run", lambda: assert_true({name:sha256(path) for name,path in protected_paths.items()}==protected_before))
failed = [x for x in test_results if not x[1]]
assert not failed, failed
print(f"TESTS PASS — {len(test_results)}/{len(test_results)}")


In [ ]:
# 8. Decisions και regression cases
decision_rows = []
if selected_gate:
    for d in selected_gate["decisions"]:
        s=d["signals"]; c=s["candidate"] if s else None; key=(d["development_index"],c["evidence_id"]) if c else None
        decision_rows.append({"development_index":d["development_index"],"term":term_by_index[d["development_index"]],"query_variant":selected_gate["query_variant"],"reference_assisted":selected_gate["reference_assisted"],"retrieval_method":selected_gate["retrieval_method"],"gate_decision":"accept" if d["accepted"] else "abstain","abstention_reasons":d["reasons"],"evidence_id":c["evidence_id"] if c else None,"method_rank":c["method_rank"] if c else None,"score":s["score"] if s else None,"relative_margin":s["relative_margin"] if s else None,"method_agreement":s["method_agreement"] if s else None,"exact_match":s["exact_match"] if s else None,"human_relevance_category":d["label"],"annotation_confidence":annotation_meta[key]["confidence"] if key else None})
else:
    decision_rows=[{"development_index":idx,"term":term_by_index[idx],"gate_decision":"not_selected","status":"NO_GO_FOR_HOLDOUT"} for idx in development_indices]

regression_terms={"preciosity","dateliner","continuous","weary","inconsistent"}
regression_rows=[]
for d in decision_rows:
    if d["term"].lower() in regression_terms:
        passed=d["gate_decision"]=="abstain" or (d["gate_decision"]=="accept" and d.get("human_relevance_category") in STRICT)
        regression_rows.append({**d,"regression_pass":passed})
regression_rows


In [ ]:
# 9. Atomic output writers και results report
def atomic_write_text(path, text, encoding="utf-8"):
    CTX.write_text(path, text, encoding)

def csv_text(rows, fields):
    import io
    buf=io.StringIO(newline="")
    writer=csv.DictWriter(buf,fieldnames=fields,lineterminator="\n",extrasaction="ignore")
    writer.writeheader()
    for row in rows:
        writer.writerow({k:("NA" if row.get(k) is None else row.get(k)) for k in fields})
    return buf.getvalue()

metric_fields=list(metric_rows[0])
sensitivity_public=[{k:v for k,v in row.items() if k!="decisions"} for row in sensitivity_rows]
sensitivity_fields=list(sensitivity_public[0])
atomic_write_text(OUTPUTS["metrics"],csv_text(metric_rows,metric_fields),"utf-8")
atomic_write_text(OUTPUTS["sensitivity"],csv_text(sensitivity_public,sensitivity_fields),"utf-8")
atomic_write_text(OUTPUTS["decisions"],"".join(json.dumps(r,ensure_ascii=False,sort_keys=True)+"\n" for r in decision_rows),"utf-8")

selected_payload={"phase":"Module 2 Phase B preliminary development selection","status":selection_status,"random_state":42,"development_examples":10,"human_annotation_sha256":EXPECTED_HUMAN_SHA256,"query_variant_note":{"Q_SRC":"term plus bounded source context","Q_SRC_TGT":"reference-assisted; not independent external validation"},"selection_rule":{"minimum_accepted_count":2,"minimum_lenient_usable_coverage":0.20,"maximum_unsafe_evidence_rate_accepted":0.10,"tie_break":"unsafe accepted asc, lenient usable desc, unsafe all asc, strict usable desc, evidence coverage asc, configuration_id asc"},"selected_configuration":({k:v for k,v in selected_gate.items() if k not in {"decisions"}} if selected_gate else None),"holdout_gate_frozen":bool(selected_gate),"limitations":["Only 10 retrieval-development examples.","Only 3 candidate rows are strict relevant.","Q_SRC_TGT is reference-assisted.","Selection is preliminary and must not be interpreted as external validation."]}
atomic_write_text(OUTPUTS["selected"],json.dumps(selected_payload,ensure_ascii=False,indent=2,sort_keys=True)+"\n","utf-8")

def pct(v): return "NA" if v is None else f"{100*v:.1f}%"
lines=["# SHROOM RAG E5 — Module 2 Phase B Results","","## Scope","","Offline evaluation using only locked human relevance annotations. `Q_SRC_TGT` is reference-assisted and is not independent external validation.","","## Preflight and tests","",f"- Preflight checks: {len(checks)}/{len(checks)} passed",f"- Deterministic tests: {len(test_results)}/{len(test_results)} passed","- Human annotations: 83 rows, 83 unique IDs, 10 development examples",f"- Stable Phase A appearance joins: {joined_appearances}/192",f"- Annotation SHA-256: `{EXPECTED_HUMAN_SHA256}`","","## Retrieval metrics","","| Query | Method | Strict R@1 | Lenient R@1 | Unsafe top-1 | Eligible@3 |","|---|---|---:|---:|---:|---:|"]
for r in metric_rows: lines.append(f"| {r['query_variant']} | {r['retrieval_method']} | {pct(r['strict_relevant_at_1'])} | {pct(r['lenient_relevant_at_1'])} | {pct(r['unsafe_top1_rate'])} | {pct(r['evidence_eligible_coverage_at_3'])} |")
lines += ["","Precision@4/@5 and full MRR are `NA` wherever retrieved positions are not fully annotated. MRR@3 is retained in the CSV only as an explicitly truncated diagnostic.","","## Gate sensitivity and Pareto frontier","",f"- Deterministic grid configurations: {len(sensitivity_rows)}",f"- Non-trivial Pareto configurations: {len(pareto_rows)}",f"- Selection status: **{selection_status}**"]
if selected_gate:
    lines += [f"- Configuration: `{selected_gate['configuration_id']}`",f"- Evidence coverage: {pct(selected_gate['evidence_coverage'])}",f"- Strict / lenient usable coverage: {pct(selected_gate['strict_usable_coverage'])} / {pct(selected_gate['lenient_usable_coverage'])}",f"- Unsafe evidence rate (accepted / all): {pct(selected_gate['unsafe_evidence_rate_accepted'])} / {pct(selected_gate['unsafe_evidence_rate_all'])}",f"- Abstention rate: {pct(selected_gate['abstention_rate'])}"]
else: lines += ["","`NO_GO_FOR_HOLDOUT` — no non-trivial configuration met the predeclared conservative development criteria."]
lines += ["","### Pareto frontier (unique operating points)","","| Usable lenient | Unsafe/all | Abstention | Example configuration |","|---:|---:|---:|---|"]
seen=set()
for r in sorted(pareto_rows,key=lambda x:(-x['lenient_usable_coverage'],x['unsafe_evidence_rate_all'],x['abstention_rate'],x['configuration_id'])):
    point=(r['lenient_usable_coverage'],r['unsafe_evidence_rate_all'],r['abstention_rate'])
    if point not in seen: lines.append(f"| {pct(point[0])} | {pct(point[1])} | {pct(point[2])} | `{r['configuration_id']}` |"); seen.add(point)
lines += ["","## Regression cases","","| Term | Decision | Human category | Pass |","|---|---|---|---|"]
for r in regression_rows: lines.append(f"| {r['term']} | {r['gate_decision']} | {r.get('human_relevance_category','NA')} | {r['regression_pass']} |")
lines += ["","## Limitations","","- Ten development examples are insufficient for a stable estimate or generalization claim.","- Only three annotated candidates are strict relevant; one annotation has medium confidence.","- The grid is tuned on the same development subset used for reporting and therefore is optimistic.","- `uncertain_review` is neither relevant nor unsafe; none was forced into either class.","- Holdout remains sealed. No Gemini/API, retrieval, dense encoding, download, package change, official test, Module 3, or E3 integration was performed.",""]
atomic_write_text(OUTPUTS["report"],"\n".join(lines),"utf-8")
print("Atomic outputs written:",len(OUTPUTS))


In [ ]:
# 10. Τελική offline validation μετά τις atomic εγγραφές
for path in OUTPUTS.values(): assert path.exists() and path.stat().st_size > 0, path
written_metrics,_=read_csv(OUTPUTS["metrics"])
written_sensitivity,_=read_csv(OUTPUTS["sensitivity"])
written_decisions=read_jsonl(OUTPUTS["decisions"])
written_selected=json.loads(OUTPUTS["selected"].read_text(encoding="utf-8"))
assert len(written_metrics)==8
assert len(written_sensitivity)==len(sensitivity_rows)
assert len(written_decisions)==10
assert written_selected["status"]==selection_status
assert {name:sha256(path) for name,path in protected_paths.items()}==protected_before
assert all(sha256(CTX.evidence("RAG/results/rag_module2/" + rel))==expected.lower() for rel,expected in pool_manifest["phase_a_tree_before_creation"]["files"].items())
artifact_hashes={path.name:sha256(path) for path in OUTPUTS.values()}
print("FINAL VALIDATION PASS")
print(json.dumps(artifact_hashes,indent=2,sort_keys=True))


## Stop condition

Η Phase B σταματά εδώ. Δεν ξεκινά holdout annotation/evaluation, Gemini, Module 3 ή ενσωμάτωση στο E3.
